PART 1: **SERVER** CODE

In [22]:
# Server
import socket
import threading
import queue

def handle_client(conn, addr, q):
    print(f"[Server] Connected with: {addr}")
    stop_command = False
    try:
        while True:
            data = conn.recv(1024)
            if not data:
                break
            message = data.decode()
            print(f"[Server] Received a message from client: {message}")
            conn.sendall(b"Message received by server\n")
            stop_command = (message == "/STOP")
    except Exception as e:
        print(f"Error with client {addr}: {e}")
    finally:
        conn.close()
        print(f"[Server] Client {addr} disconnected")
    q.put(stop_command)

def run_server(host, port):
    #host = '0.0.0.0' # Bind to all available interfaces
    #port = 8000 # Use a port above 1024 if possible

    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server_socket.bind((host, port))
    server_socket.listen(5)
    print(f"Server listening on {host}:{port}")

    q = queue.Queue()
    try:
        stopped = False
        while not stopped:
            conn, addr = server_socket.accept()
            client_thread = threading.Thread(target=handle_client, args=(conn, addr, q))
            client_thread.start()
            stopped = q.get()

    except KeyboardInterrupt:
        print("Server is interrupted. Shutting down..")
    finally:
        print("Server is shutting down.")
        server_socket.close()

# Run the server in a separate thread so the Colab cell doesn't block indefinitely
server_thread = threading.Thread(target=run_server, args=('0.0.0.0', 8000))
server_thread.start()

Server listening on 0.0.0.0:8000


Client-side code

In [21]:
import socket

HOST = "0.0.0.0" # The server's hostname or IP address
PORT = 8000        # The port used by the server

# Create a socket object
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.connect((HOST, PORT)) # Connect to the server
    s.sendall(b"Hello, world!") # Send data to the server (must be bytes)
    # Send STOP command to shutdown the server
    #s.sendall(b"/STOP")
    data = s.recv(1024) # Receive the echoed data back

print(f"Received: {data.decode('utf-8')}") # Decode bytes to a string

[Server] Connected with: ('127.0.0.1', 59782)
[Server] Received a message from client: /STOP
Received: Message received by server

[Server] Client ('127.0.0.1', 59782) disconnected
Server is shutting down.


PART 2: Publish-Subscribe

In [2]:
!pip install pypubsub

In [3]:
"""
One listener is subscribed to a topic called 'rootTopic'.
One 'rootTopic' message gets sent.
"""
from pubsub import pub

# Create a listener
def listener1(arg1, arg2=None):
    print('Function listener1 received:')
    print('  arg1 =', arg1)
    print('  arg2 =', arg2)

# Register listener
pub.subscribe(listener1, 'rootTopic')

# Send a message
print('Publish something via pubsub')
anObj = dict(a=456, b='abc')
pub.sendMessage('rootTopic', arg1=123, arg2=anObj)

Publish something via pubsub
Function listener1 received:
  arg1 = 123
  arg2 = {'a': 456, 'b': 'abc'}


In [4]:
anObj2 = dict(a=789, b='cde')
pub.sendMessage('rootTopic', arg1=123, arg2=anObj2)

Function listener1 received:
  arg1 = 123
  arg2 = {'a': 789, 'b': 'cde'}


**Assignment**:
Buat sebuah listener dengan nama listener2 yang di-subscribe ke rootTopic, yang melakukan sbb:

*   Menjalankan print untuk **arg2**
*   Melempar nilai di **arg1** ke server yang telah dibuat di PART 1 (pastikan server jalan).
*   Buat laporan (PDF) berisi kode dan penjelasan

